## Ejercicio 3.
Utilizando las funciones provistas por Scikit-learn, implememente los métodos de ensambles de clasificadores Bagging y AdaBoost. Compare el desempeño de estos modelos empleando 5 particiones con el conjunto de datos Wine.


In [2]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn import tree
from sklearn import svm

X, y = datasets.load_wine(return_X_y=True)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

estimadores = {
    'Tree': tree.DecisionTreeClassifier(criterion='gini', random_state=42),
    'GaussianNB': GaussianNB(),
    'ADL': LinearDiscriminantAnalysis(),
    'KNN': KNeighborsClassifier(n_neighbors=3, weights='distance', algorithm='auto', p=2, metric='minkowski'),
    'SVM': svm.SVC(kernel='sigmoid', random_state=42)
}

numeros_estimadores = [10, 30, 50]

for nombre, estimador_base in estimadores.items():
    print(f"\nModelo: {nombre}")
    for n_est in numeros_estimadores:
        scores_bagging = []
        scores_adaboost = []
        for i_trn, i_tst in kf.split(X):
            X_trn, X_tst = X[i_trn], X[i_tst]
            y_trn, y_tst = y[i_trn], y[i_tst]

            scaler = StandardScaler()
            X_trn = scaler.fit_transform(X_trn)
            X_tst = scaler.transform(X_tst)

            # BAGGING
            clf_bag = BaggingClassifier(estimator=estimador_base, n_estimators=n_est, random_state=42)
            clf_bag.fit(X_trn, y_trn)
            y_pred = clf_bag.predict(X_tst)
            scores_bagging.append(accuracy_score(y_tst, y_pred))

            # ADABOOST
            try:
                clf_ada = AdaBoostClassifier(estimator=estimador_base, n_estimators=n_est, random_state=42)
                clf_ada.fit(X_trn, y_trn)
                y_pred = clf_ada.predict(X_tst)
                scores_adaboost.append(accuracy_score(y_tst, y_pred))
            except Exception:
                pass
            
        print(f"Bagging (num_est={n_est}): Media {np.mean(scores_bagging):.4f} | Desvío {np.std(scores_bagging):.4f}")
        if scores_adaboost:
            print(f"AdaBoost (num_est{n_est}): Media {np.mean(scores_adaboost):.4f} | Desvío {np.std(scores_adaboost):.4f}")
        else:
            #el clasificador no soporta ponderación de muestras (KNN - ADL)
            print(f"AdaBoost (num_est={n_est})-> No soporta: Error('sample_weight')") 


Modelo: Tree
Bagging (num_est=10): Media 0.9719 | Desvío 0.0252
AdaBoost (num_est10): Media 0.8765 | Desvío 0.0571
Bagging (num_est=30): Media 0.9606 | Desvío 0.0224
AdaBoost (num_est30): Media 0.8765 | Desvío 0.0571
Bagging (num_est=50): Media 0.9776 | Desvío 0.0208
AdaBoost (num_est50): Media 0.8765 | Desvío 0.0571

Modelo: GaussianNB
Bagging (num_est=10): Media 0.9830 | Desvío 0.0228
AdaBoost (num_est10): Media 0.9887 | Desvío 0.0138
Bagging (num_est=30): Media 0.9830 | Desvío 0.0228
AdaBoost (num_est30): Media 0.9887 | Desvío 0.0138
Bagging (num_est=50): Media 0.9887 | Desvío 0.0138
AdaBoost (num_est50): Media 0.9887 | Desvío 0.0138

Modelo: ADL
Bagging (num_est=10): Media 0.9944 | Desvío 0.0111
AdaBoost (num_est=10)-> No soporta: Error('sample_weight')
Bagging (num_est=30): Media 0.9887 | Desvío 0.0138
AdaBoost (num_est=30)-> No soporta: Error('sample_weight')
Bagging (num_est=50): Media 0.9887 | Desvío 0.0138
AdaBoost (num_est=50)-> No soporta: Error('sample_weight')

Modelo: KN